### Importing of libraries

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
import sqlite3
%matplotlib inline
import math
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l1_l2
import warnings
warnings.filterwarnings('ignore')

### Data Import

In [2]:
db_path = r"E:\Machine Learning Research\DataModelling\IDA_Data.db"  # <-- change this to your actual .db file path
table_name = "IDA_Data"                    # <-- your table name

conn = sqlite3.connect(db_path)
df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
conn.close()

# === Display===
# print(df.shape)
# print(list(df.columns))
# === Optionally, show first few rows ===
# print(InputHeads)
# print(OutputHeads)
# print("\nFirst 5 rows:")
# print(df.columns)
# df.columns

AllColumns = ['id', 'Earthquake', 'ScaleFactor', 'Building', 'BaseCondition', 'ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx', 'ly-lx-', 'lx-ly-', 'lx-ly--ly-lx-', 'Plan-area', 'Seismic-weight', 
              'StiffnessX_Story5', 'StiffnessX_Story4', 'StiffnessX_Story3', 'StiffnessX_Story2', 'StiffnessX_Story1', 'StiffnessX_Total', 'Layer1_FrictionA', 'Layer1_G_kPa', 
              'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion', 'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion',
              'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion', 'PGA', 'Magnitude', 'Mechanism', 'Rjb', 'Rrup', 'Vs30', 'cav_gs', 
              'scav_gs', 'bcav_gs', 'arias_mps', 'husid_s', 'spi_mps', 'hous_m', 'maxacceleration_mps2', 'maxvelocity_mps', 'maxdisplacement_m', 'maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps', 
              'Fundamental_Period', 'Drift-X_Level-1', 'Drift-X_Level-2', 'Drift-X_Level-3', 'Drift-X_Level-4', 'Drift-X_Level-5', 'Drift-X_Level-6', 'Drift-Y_Level-1', 'Drift-Y_Level-2',
              'Drift-Y_Level-3', 'Drift-Y_Level-4', 'Drift-Y_Level-5', 'Drift-Y_Level-6', 'Displacement-X_Level-1', 'Displacement-X_Level-2', 'Displacement-X_Level-3', 'Displacement-X_Level-4',
              'Displacement-X_Level-5', 'Displacement-X_Level-6', 'Displacement-Y_Level-1', 'Displacement-Y_Level-2', 'Displacement-Y_Level-3', 'Displacement-Y_Level-4', 'Displacement-Y_Level-5',
              'Displacement-Y_Level-6', 'Reaction-Force-X_Level-1', 'Reaction-Force-X_Level-2', 'Reaction-Force-X_Level-3', 'Reaction-Force-X_Level-4', 'Reaction-Force-X_Level-5', 'Reaction-Force-X_Level-6', 
              'Reaction-Force-Y_Level-1', 'Reaction-Force-Y_Level-2', 'Reaction-Force-Y_Level-3', 'Reaction-Force-Y_Level-4', 'Reaction-Force-Y_Level-5', 'Reaction-Force-Y_Level-6', 'Reaction-Moment-X_Level-1',
              'Reaction-Moment-X_Level-2', 'Reaction-Moment-X_Level-3', 'Reaction-Moment-X_Level-4', 'Reaction-Moment-X_Level-5', 'Reaction-Moment-X_Level-6', 'Reaction-Moment-Y_Level-1',
              'Reaction-Moment-Y_Level-2', 'Reaction-Moment-Y_Level-3', 'Reaction-Moment-Y_Level-4', 'Reaction-Moment-Y_Level-5', 'Reaction-Moment-Y_Level-6', 'Rotation-X_Level-1', 'Rotation-X_Level-2',
              'Rotation-X_Level-3', 'Rotation-X_Level-4', 'Rotation-X_Level-5', 'Rotation-X_Level-6', 'Rotation-Y_Level-1', 'Rotation-Y_Level-2', 'Rotation-Y_Level-3', 'Rotation-Y_Level-4', 'Rotation-Y_Level-5',
              'Rotation-Y_Level-6', 'Rotation-Z_Level-1', 'Rotation-Z_Level-2', 'Rotation-Z_Level-3', 'Rotation-Z_Level-4', 'Rotation-Z_Level-5', 'Rotation-Z_Level-6', 'Torsional-Irregularity-Ratio_Level-1',
              'Torsional-Irregularity-Ratio_Level-2', 'Torsional-Irregularity-Ratio_Level-3', 'Torsional-Irregularity-Ratio_Level-4', 'Torsional-Irregularity-Ratio_Level-5', 'Torsional-Irregularity-Ratio_Level-6',
              'Max-Uplift_Level-1', 'Max-Uplift-Point_Level-1', 'Max-Settlement_Level-1', 'Max-Settlement-Point_Level-1', 'Max-Pseudo-Time_Level-1']

### Data Processing

In [3]:
def dataPreparation(DoNormalize = True,Z_ScoreScaler = True    ,        ModelSet = 2     ,includeCategoricalData = False  ):
    
    # Creating of the PeakGroundAcceleration column on the database by product of ScaleFactor and MaximumAcceleration of the data
    df['maxacceleration_mps2'] = pd.to_numeric(df['maxacceleration_mps2'], errors='coerce')
    df['ScaleFactor'] = pd.to_numeric(df['ScaleFactor'], errors='coerce')
    df['PeakGroundAcceleration'] = df['maxacceleration_mps2'] * df['ScaleFactor']
    
    #Creating of the Prediction column as Max drift of the all levels merger in single column as below
    drift_cols = [    'Drift-X_Level-1', 'Drift-X_Level-2', 'Drift-X_Level-3',    'Drift-X_Level-4', 'Drift-X_Level-5', 'Drift-X_Level-6']
    df[drift_cols] = df[drift_cols].apply(pd.to_numeric, errors='coerce')
    df['Max_Drift_X'] = df[drift_cols].max(axis=1)
    
    # Check for any NaNs that might have appeared due to conversion issues
    # print(df[['maxacceleration_mps2', 'ScaleFactor', 'PeakGroundAcceleration']].sample(50))
    # print("Number of NaNs in new column:", df['PeakGroundAcceleration'].isna().sum())
    
    InputHeadsAvailableAll = ['Earthquake', 'ScaleFactor', 'Building', 'BaseCondition', 'ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx', 'ly-lx-', 'lx-ly-', 'lx-ly--ly-lx-', 'Plan-area', 'Seismic-weight', 
                  'StiffnessX_Story5', 'StiffnessX_Story4', 'StiffnessX_Story3', 'StiffnessX_Story2', 'StiffnessX_Story1', 'StiffnessX_Total', 
                  'Layer1_FrictionA', 'Layer1_G_kPa', 'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion',
                  'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion', 
                  'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion', 
                  'PGA', 'Magnitude', 'Mechanism', 'Rjb', 'Rrup', 'Vs30', 'cav_gs', 'scav_gs', 'bcav_gs', 'arias_mps', 'husid_s', 'spi_mps', 'hous_m', 'maxacceleration_mps2',
                  'maxvelocity_mps', 'maxdisplacement_m', 'maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps', 
                  'Fundamental_Period']
    InputHeadsSelected = ['ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx', 'ly-lx-', 'lx-ly-', 'lx-ly--ly-lx-',
                   'StiffnessX_Total', 'Plan-area', 'Seismic-weight', 
                  'Layer1_FrictionA', 'Layer1_G_kPa', 'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion',
                  'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion', 
                  'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion', 
                          
                  "PeakGroundAcceleration", 'Magnitude', 'arias_mps','maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps',  'cav_gs', 
                             'Fundamental_Period']
    # InputHeadsSelected = InputHeadsAvailableAll

    
    #Best performer for the fixedbase and displacement output
                   # "PeakGroundAcceleration", 'Magnitude', 'arias_mps','maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps',  'cav_gs', 
                   #           'Fundamental_Period']
    #Best performer for the maximum drift along X with r2 as 89.36 (Donot include  'Plan-area', 'Seismic-weight', 'StiffnessX_Total', as they donot produce any effect and only add burden to model)
    # InputHeadsSelected = ['ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx', 'ly-lx-', 'lx-ly-', 'lx-ly--ly-lx-',
                   
    #               'Layer1_FrictionA', 'Layer1_G_kPa', 'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion',
    #               'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion', 
    #               'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion', 
                          
    #               "PeakGroundAcceleration", 'Magnitude', 'arias_mps','maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps',  'cav_gs', 
    #                          'Fundamental_Period']
    #Best performer for the fixed and output as Reaction-Force-X_Level-1:  r2 as 0.95 max with 15 features
    # InputHeadsSelected = ['Earthquake', 'Building', 'Fundamental_Period',  'ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx',
    #               'Layer1_FrictionA', 'Layer1_G_kPa', 'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion',
    #               'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion', 
    #               'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion', 
    #                "PeakGroundAcceleration", 'arias_mps','maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps',
    #                         ]
    #For soil analysis take Rup, Vs30... Use maxacceleration_mps2 that represents the pga value of site and for the building specific analysis ie performance
    # Levels measurement take PSA, PSV and PSD values over the peac acc peak vel and peak disp and so on 
    #Final selected features (9): ['ScaleFactor', 'Vs30', 'cav_gs', 'arias_mps', 'husid_s', 'hous_m', 'maxvelocity_mps', 'maxdisplacement_m', 'maxpsd_cmps']
    
    
    
    #_______________________________________________________________________________________________________________________________________________________________Processing starts from here
    if ModelSet == 1 :
        InputHeads = [x for x in InputHeadsSelected if not x.startswith("Layer")] 
    elif ModelSet == 3:
        InputHeads = [x for x in InputHeadsSelected if not x.startswith("Layer")]
        InputHeads = InputHeads + ["BaseCondition"]
    else:
        InputHeads = InputHeadsSelected
    # OutputHeads = ['Displacement-X_Level-5']
    # OutputHeads = ["Reaction-Force-X_Level-1"]
    OutputHeads = ['Max_Drift_X']
    
    # df[InputHeads].info
    
    ####__________________________________________________________________________________________________________________________________Data selection 
    
    # Extract all the data from the fixed base conditions only
    if ModelSet == 1:
        df_fixed = df[df['BaseCondition'] == 'Fixed']
    elif ModelSet == 2:
        df_fixed = df[df['BaseCondition'] != 'Fixed']
    else:
        df_fixed = df[df['BaseCondition'].isin(['Fixed', 'Soft', 'Medium', 'Hard'])]
        
    
    #Convert all the data in database to numerical format
    if includeCategoricalData:
        df_numeric = df_fixed.copy()
        for col in df_numeric.columns:
            df_numeric[col] = pd.to_numeric(df_numeric[col], errors='ignore')
    else:
        df_numeric = df_fixed.apply(pd.to_numeric, errors='coerce')
        df_numeric.shape
    # df_numeric.head
    
    ##Extract the data if the scale factor is less than or equal to 1 only
    if ModelSet != 1:
        df_numeric = df_numeric[df_numeric['ScaleFactor'] <= 1]
    df_numeric = df_numeric[df_numeric['ScaleFactor'] <= 1]
    
    #Removing all the white spaces on the column titles
    df_numeric.columns = df_numeric.columns.str.replace(" ", "_")

    ####__________________________________________________________________________________________________________________________________Encoding of data
    # # Normalize for the data of the features as seismic weights, and so on
    features_to_normalize = [x for x in InputHeads
                             if pd.api.types.is_numeric_dtype(df_numeric[x])]
    
    
    #One hot encoding of the categorical data if user set it to include in the model
    if includeCategoricalData:
        Catcolumns =  [x for x in InputHeads if x not in features_to_normalize]
        # print(Catcolumns)

        if Catcolumns:
            data_toEncode = df_numeric[Catcolumns]     
            CatDataEncoded = pd.get_dummies(data_toEncode, columns = Catcolumns,  dtype=float)
            df_numeric = df_numeric.drop(columns=Catcolumns)

        
            if df_numeric.shape[0] == CatDataEncoded.shape[0]:
                InputHeads = InputHeads + list(CatDataEncoded.columns)
                df_numeric = pd.concat([df_numeric, CatDataEncoded], axis=1)

            else: 
                raise
    

    
    ## Converts all the data to the numeric even to that of hot encoded as they are in float
    df_numeric = df_numeric.apply(pd.to_numeric, errors='coerce')
        
    
    ####__________________________________________________________________________________________________________________________________Missing Data Handling
    df_numeric =df_numeric.dropna(axis=1, how='all')             #Dropping for all the columns (axis = 1) if it has all values of NaN
    
    #### Show only columns that actually contain NaN
    # nan_counts = df_numeric.isna().sum()
    # nan_counts = nan_counts[nan_counts > 0]
    # print("Columns containing NaN values (column: count):")
    # print(nan_counts)
    
    # nan_rows = df_numeric[df_numeric.isna().any(axis=1)]
    # print("Rows containing NaN values:")
    # print(nan_rows)
    
    df_numeric = df_numeric.apply(lambda col: col.fillna(col.median()) if col.dtype != 'object' else col)   #For any value other than object if the data is shown as missing then it fills up with the median of that column

    # nan_counts = df_numeric.isna().sum()
    # nan_counts = nan_counts[nan_counts > 0]
    # print("Columns containing NaN values (column: count):")
    # print(nan_counts)
    
    df_numeric = df_numeric.dropna()                             #Now also if any of the Nan data persists then the system drops that specific rows
    

    ####__________________________________________________________________________________________________________________________________Normalization of data
    if DoNormalize:
        scaler = MinMaxScaler(feature_range=(0, 1))
        if Z_ScoreScaler:
            scaler = StandardScaler()
        cols_to_scale = [c for c in features_to_normalize if c in df_numeric.columns] # Only normalize columns that actually exist in df
        df_numeric[cols_to_scale] = scaler.fit_transform(df_numeric[cols_to_scale])

    ####__________________________________________________________________________________________________________________________________Assignment of the training data
    
    # # Extract for the data to be in x and y variables
    X = df_numeric[[col for col in InputHeads if col in df_numeric.columns]]
    y = df_numeric[[col for col in OutputHeads if col in df_numeric.columns]]

    df_numeric = pd.concat([X, y], axis=1)  # combine X and y

    
    # X.shape
    # print((X.columns))
    # cols = CatDataEncoded.columns
    # for co in cols:
    #     print(co)
    # print(CatDataEncoded.columns)
    # df_numeric.sample(10)
    # X.shape

    return X, y



### Getting data Statistics

In [4]:
import pandas as pd
import numpy as np

def generate_complete_data_statistics(X, y):
    """Generate comprehensive statistical summary for entire dataset"""
    
    # Convert to proper formats
    X_clean = X.apply(pd.to_numeric, errors='coerce')
    y_clean = y.values.ravel() if hasattr(y, 'values') else y
    
    print("COMPLETE DATASET STATISTICAL SUMMARY")
    print("="*80)
    print(f"{'Dataset':<20} {'Samples':<10} {'Features':<10}")
    print(f"{'':<20} {X_clean.shape[0]:<10} {X_clean.shape[1]:<10}")
    print("="*80)
    
    # Features summary
    print(f"\n{'FEATURES SUMMARY':<50}")
    print("-"*50)
    print(f"{'Parameter':<25} {'Count':<8} {'Mean':<12} {'Std':<12} {'Min':<10} {'Max':<10}")
    print("-"*50)
    
    for column in X_clean.columns:
        count = len(X_clean[column])
        mean = X_clean[column].mean()
        std = X_clean[column].std()
        min_val = X_clean[column].min()
        max_val = X_clean[column].max()
        
        print(f"{column:<25} {count:<8} {mean:<12.4f} {std:<12.4f} {min_val:<10.2f} {max_val:<10.2f}")

    # Target variable summary
    print(f"\n{'TARGET VARIABLE SUMMARY':<50}")
    print("-"*50)
    print(f"{'Parameter':<25} {'Count':<8} {'Mean':<12} {'Std':<12} {'Min':<10} {'Max':<10}")
    print("-"*50)
    
    count = len(y_clean)
    mean = y_clean.mean()
    std = y_clean.std()
    min_val = y_clean.min()
    max_val = y_clean.max()
    
    print(f"{'Target':<25} {count:<8} {mean:<12.4f} {std:<12.4f} {min_val:<10.2f} {max_val:<10.2f}")

    # Additional statistics
    print(f"\n{'ADDITIONAL STATISTICS':<50}")
    print("-"*50)
    print(f"{'Metric':<25} {'Value':<25}")
    print("-"*50)
    print(f"{'Total Samples':<25} {X_clean.shape[0]:<25}")
    print(f"{'Total Features':<25} {X_clean.shape[1]:<25}")
    print(f"{'Missing Values in X':<25} {X_clean.isnull().sum().sum():<25}")
    print(f"{'Missing Values in y':<25} {np.isnan(y_clean).sum():<25}")
    print(f"{'Target Skewness':<25} {pd.Series(y_clean).skew():<25.4f}")
    print(f"{'Target Kurtosis':<25} {pd.Series(y_clean).kurtosis():<25.4f}")

# Quick version for compact output
def quick_data_stats(X, y):
    """Quick statistical summary"""
    X_clean = X.apply(pd.to_numeric, errors='coerce')
    y_clean = y.values.ravel() if hasattr(y, 'values') else y
    
    print("DATASET STATISTICS")
    print("="*90)
    print(f"{'Parameter':<20} {'Count':<8} {'Mean':<12} {'Std':<12} {'Min':<10} {'Max':<10} {'Type':<10}")
    print("-"*90)
    
    # Features
    for column in X_clean.columns:
        data = X_clean[column]
        print(f"{column:<20} {len(data):<8} {data.mean():<12.4f} {data.std():<12.4f} "
              f"{data.min():<10.2f} {data.max():<10.2f} {'Feature':<10}")
    
    # Target
    print(f"{'Target':<20} {len(y_clean):<8} {y_clean.mean():<12.4f} {y_clean.std():<12.4f} "
          f"{y_clean.min():<10.2f} {y_clean.max():<10.2f} {'Target':<10}")

# Generate LaTeX table for research paper
def generate_latex_table(X, y):
    """Generate LaTeX table for research paper"""
    X_clean = X.apply(pd.to_numeric, errors='coerce')
    y_clean = y.values.ravel() if hasattr(y, 'values') else y
    
    print("\nLATEX TABLE FOR RESEARCH PAPER:")
    print("\\begin{table}[htbp]")
    print("\\centering")
    print("\\caption{Descriptive Statistics of the Complete Dataset}")
    print("\\label{tab:data_statistics}")
    print("\\begin{tabular}{lrrrrr}")
    print("\\hline")
    print("Parameter & Count & Mean & Std & Min & Max \\\\")
    print("\\hline")
    
    # Features
    for column in X_clean.columns:
        data = X_clean[column]
        print(f"{column} & {len(data)} & {data.mean():.4f} & {data.std():.4f} & {data.min():.2f} & {data.max():.2f} \\\\")
    
    # Target
    print(f"Target & {len(y_clean)} & {y_clean.mean():.4f} & {y_clean.std():.4f} & {y_clean.min():.2f} & {y_clean.max():.2f} \\\\")
    print("\\hline")
    print("\\end{tabular}")
    print("\\end{table}")



# Neural Network Final

In [5]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer

class EnhancedNeuralNetworkAnalyzer:
    def __init__(self, X, y, test_size=0.2, random_state=42, 
                 augment_data=False, noise_factor=0.01, stratify_data=False, n_bins=5):
        """Initialize enhanced neural network analyzer with augmentation and stratification"""
        self.X = X.copy()
        self.y = y.copy()
        self.test_size = test_size
        self.random_state = random_state
        self.augment_data = augment_data
        self.noise_factor = noise_factor
        self.stratify_data = stratify_data
        self.n_bins = n_bins
        self.scaler = StandardScaler()
        self.history = None
        self.model = None
        
        # Prepare data with enhanced features
        self._prepare_enhanced_data()
        
        print(f"Enhanced Neural Network Analyzer initialized:")
        print(f"Training data: {self.X_train_scaled.shape}")
        print(f"Testing data: {self.X_test_scaled.shape}")
        print(f"Data augmentation: {self.augment_data}")
        print(f"Stratified splitting: {self.stratify_data}")

    
    
    def _create_stratification_bins(self, y, n_bins=5):
        """Create bins for continuous target variable for stratification"""
        # Use KBinsDiscretizer to create balanced bins
        discretizer = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile')
        y_binned = discretizer.fit_transform(y.reshape(-1, 1)).ravel()
        return y_binned.astype(int)
    
    def add_data_augmentation(self, X_train, y_train, noise_factor=0.01):
        """Add Gaussian noise to training data for regularization"""
        print(f"🔧 Applying data augmentation with noise factor: {noise_factor}")
        
        # Add Gaussian noise
        X_noise = X_train + noise_factor * np.random.normal(
            loc=0.0, scale=1.0, size=X_train.shape
        )
        
        # Combine original and augmented data
        X_augmented = np.vstack([X_train, X_noise])
        y_augmented = np.concatenate([y_train, y_train])
        
        print(f"📈 Data augmented: {X_train.shape} -> {X_augmented.shape}")
        
        return X_augmented, y_augmented
    
    def _prepare_enhanced_data(self):
        """Enhanced data preparation with stratification and augmentation"""
        try:
            # Convert to numpy arrays
            if hasattr(self.X, 'values'):
                X_values = self.X.values
            else:
                X_values = np.array(self.X)
                
            if hasattr(self.y, 'values'):
                y_values = self.y.values
            else:
                y_values = np.array(self.y)
            
            y_values = y_values.ravel()
            
            # Stratified splitting for continuous target
            if self.stratify_data:
                print("🎯 Using stratified data splitting...")
                # Create bins for stratification
                y_binned = self._create_stratification_bins(y_values, self.n_bins)
                
                # Use stratified split
                strat_split = StratifiedShuffleSplit(
                    n_splits=1, 
                    test_size=self.test_size, 
                    random_state=self.random_state
                )
                
                for train_idx, test_idx in strat_split.split(X_values, y_binned):
                    self.X_train, self.X_test = X_values[train_idx], X_values[test_idx]
                    self.y_train, self.y_test = y_values[train_idx], y_values[test_idx]
                    
                print(f"📊 Stratification bins: {self.n_bins}")
                print(f"📈 Target distribution in train: {np.unique(y_binned[train_idx], return_counts=True)}")
                print(f"📊 Target distribution in test: {np.unique(y_binned[test_idx], return_counts=True)}")
                
            else:
                # Standard random split
                self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                    X_values, y_values, test_size=self.test_size, random_state=self.random_state
                )
            
            # Scale features
            self.X_train_scaled = self.scaler.fit_transform(self.X_train)
            self.X_test_scaled = self.scaler.transform(self.X_test)
            
            # Apply data augmentation if enabled
            if self.augment_data:
                self.X_train_scaled, self.y_train = self.add_data_augmentation(
                    self.X_train_scaled, self.y_train, self.noise_factor
                )
            
        except Exception as e:
            print(f"Error in enhanced data preparation: {e}")
            # Fallback to standard preparation
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                self.X, self.y, test_size=self.test_size, random_state=self.random_state
            )
            self.X_train_scaled = self.scaler.fit_transform(self.X_train)
            self.X_test_scaled = self.scaler.transform(self.X_test)

    def _prepare_data(self):
        """Prepare and scale the data for neural network"""
        try:
            # Convert to numpy arrays to avoid any pandas issues
            if hasattr(self.X, 'values'):
                X_values = self.X.values
            else:
                X_values = np.array(self.X)
                
            if hasattr(self.y, 'values'):
                y_values = self.y.values
            else:
                y_values = np.array(self.y)
            
            # Ensure y is 1D
            y_values = y_values.ravel()
            
            # Split the data
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                X_values, y_values, test_size=self.test_size, random_state=self.random_state
            )
            
            # Scale features
            self.X_train_scaled = self.scaler.fit_transform(self.X_train)
            self.X_test_scaled = self.scaler.transform(self.X_test)
            
        except Exception as e:
            print(f"Error in data preparation: {e}")
            # Fallback
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                self.X, self.y, test_size=self.test_size, random_state=self.random_state
            )
            self.X_train_scaled = self.scaler.fit_transform(self.X_train)
            self.X_test_scaled = self.scaler.transform(self.X_test)
    
    def create_model(self, architecture='standard', input_dim=None):
        """Create neural network model with different architectures"""
        if input_dim is None:
            input_dim = self.X_train_scaled.shape[1]
        
        model = Sequential()
        
        if architecture == 'simple':
            # Simple architecture
            model.add(Dense(64, activation='relu', input_shape=(input_dim,)))
            model.add(Dropout(0.2))
            model.add(Dense(32, activation='relu'))
            model.add(Dropout(0.2))
            model.add(Dense(1, activation='linear'))
            
        elif architecture == 'standard':
            # Standard architecture
            model.add(Dense(128, activation='relu', input_shape=(input_dim,),
                          kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
            model.add(BatchNormalization())
            model.add(Dropout(0.3))
            
            model.add(Dense(64, activation='relu', 
                          kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
            model.add(BatchNormalization())
            model.add(Dropout(0.3))
            
            model.add(Dense(32, activation='relu'))
            model.add(Dropout(0.2))
            
            model.add(Dense(1, activation='linear'))
            
        elif architecture == 'deep':
            # Deep architecture
            model.add(Dense(256, activation='relu', input_shape=(input_dim,),
                          kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
            model.add(BatchNormalization())
            model.add(Dropout(0.4))
            
            model.add(Dense(128, activation='relu',
                          kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
            model.add(BatchNormalization())
            model.add(Dropout(0.4))
            
            model.add(Dense(64, activation='relu'))
            model.add(Dropout(0.3))
            
            model.add(Dense(32, activation='relu'))
            model.add(Dropout(0.2))
            
            model.add(Dense(16, activation='relu'))
            model.add(Dropout(0.1))
            
            model.add(Dense(1, activation='linear'))
        
        elif architecture == 'wide':
            # Wide architecture
            model.add(Dense(512, activation='relu', input_shape=(input_dim,),
                          kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
            model.add(BatchNormalization())
            model.add(Dropout(0.5))
            
            model.add(Dense(256, activation='relu'))
            model.add(Dropout(0.3))
            
            model.add(Dense(128, activation='relu'))
            model.add(Dropout(0.2))
            
            model.add(Dense(1, activation='linear'))


        elif architecture == 'wide_regularized':
            # Wide architecture
            model.add(Dense(512, activation='relu', input_shape=(input_dim,),
                          kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
            model.add(BatchNormalization())
            model.add(Dropout(0.5))
            
            model.add(Dense(256, activation='relu'))
            model.add(Dropout(0.4))
            
            model.add(Dense(128, activation='relu'))
            model.add(Dropout(0.2))
            
            model.add(Dense(1, activation='linear'))
            
            # # Wide architecture
            # # Layer 1 - Much smaller with extreme regularization
            # model.add(Dense(128, activation='relu', input_shape=(input_dim,),
            #                kernel_regularizer=l1_l2(l1=1e-4, l2=5e-3)))  # 5x stronger L2
            # model.add(BatchNormalization())
            # model.add(Dropout(0.7))  # Extreme dropout
            
            # # Layer 2
            # model.add(Dense(64, activation='relu',
            #                kernel_regularizer=l1_l2(l1=1e-4, l2=2e-3)))
            # model.add(Dropout(0.6))
            
            # # Layer 3
            # model.add(Dense(32, activation='relu',
            #                kernel_regularizer=l1_l2(l1=1e-5, l2=1e-3)))
            # model.add(Dropout(0.5))
            
            # # Output
            # model.add(Dense(1, activation='linear'))
            
    
        
        return model
    
    def train_model(self, architecture='standard', epochs=500, batch_size=32, 
                   learning_rate=0.001, validation_split=0.2, verbose=0):
        """Train the neural network model"""
        print(f"Training Neural Network with {architecture} architecture...")
        print(f"Epochs: {epochs}, Batch size: {batch_size}, Learning rate: {learning_rate}")
        
        # Create model
        self.model = self.create_model(architecture=architecture)
        
        # Compile model
        optimizer = Adam(
            learning_rate=learning_rate,
            epsilon=1e-8,           # Crucial for small values
            clipnorm=1.0,           # Prevent gradient explosion

        )
        self.model.compile(
            optimizer=optimizer,
            loss='mae',
            metrics=['mae', 'mse']
        )
        
        # Callbacks
        # early_stopping = EarlyStopping(
        #     monitor='val_loss',
        #     patience=50,
        #     restore_best_weights=True,
        #     verbose=1
        # )
        
        # reduce_lr = ReduceLROnPlateau(
        #     monitor='val_loss',
        #     factor=0.5,
        #     patience=20,
        #     min_lr=1e-7,
        #     verbose=1
        # )
        
        early_stopping = EarlyStopping(
            monitor='val_loss',
            patience=30,  # Increased patience
            restore_best_weights=True,
            verbose=1,
            min_delta=0.0001  # Minimum change to qualify as improvement
        )
        
        # More aggressive learning rate reduction
        reduce_lr = ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=15,
            min_lr=1e-7,
            verbose=1
        )
        
        # Add model checkpointing
        checkpoint = tf.keras.callbacks.ModelCheckpoint(
            'best_model.h5',
            monitor='val_loss',
            save_best_only=True,
            verbose=1
        )
        #_______________________________________________________________
        
        # Train model
        self.history = self.model.fit(
            self.X_train_scaled, self.y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=validation_split,
            callbacks=[early_stopping, reduce_lr],
            verbose=verbose,
            shuffle=True
        )
        
        print("Neural Network training completed!")
        return self.history
    
    def evaluate_model(self):
        """Evaluate the trained model and return metrics"""
        if self.model is None:
            print("Model not trained yet. Please train the model first.")
            return None
        
        # Make predictions
        y_pred_train = self.model.predict(self.X_train_scaled).flatten()
        y_pred_test = self.model.predict(self.X_test_scaled).flatten()
        
        # Calculate metrics
        metrics = {
            'train_rmse': np.sqrt(mean_squared_error(self.y_train, y_pred_train)),
            'test_rmse': np.sqrt(mean_squared_error(self.y_test, y_pred_test)),
            'train_mse': mean_squared_error(self.y_train, y_pred_train),
            'test_mse': mean_squared_error(self.y_test, y_pred_test),
            'train_r2': r2_score(self.y_train, y_pred_train),
            'test_r2': r2_score(self.y_test, y_pred_test),
            'train_mae': mean_absolute_error(self.y_train, y_pred_train),
            'test_mae': mean_absolute_error(self.y_test, y_pred_test),
        }
        
        return metrics, y_pred_train, y_pred_test

# Enhanced training function
def run_enhanced_neural_network_analysis(X, y, architecture='standard', 
                                       compare_architectures=False, epochs=100, 
                                       learning_rate=0.001, batch_size=32,
                                       augment_data=False, noise_factor=0.01,
                                       stratify_data=False, n_bins=5):
    """
    Enhanced neural network analysis with data augmentation and stratification
    """
    print("🚀 Starting Enhanced Neural Network Analysis")
    print("=" * 60)
    print(f"📊 Data Augmentation: {augment_data}")
    print(f"🎯 Stratified Splitting: {stratify_data}")
    
    # Initialize enhanced analyzer
    nn_analyzer = EnhancedNeuralNetworkAnalyzer(
        X, y, 
        augment_data=augment_data,
        noise_factor=noise_factor,
        stratify_data=stratify_data,
        n_bins=n_bins
    )
    
    # Train the model
    history = nn_analyzer.train_model(
        architecture=architecture, 
        epochs=epochs, 
        learning_rate=learning_rate, 
        batch_size=batch_size
    )
    
    # Evaluate model
    metrics, _, _ = nn_analyzer.evaluate_model()
    
    print("\n" + "=" * 60)
    print("📊 ENHANCED NEURAL NETWORK RESULTS")
    print("=" * 60)
    print(f"Training R²:   {metrics['train_r2']:.4f}")
    print(f"Testing R²:    {metrics['test_r2']:.4f}")
    print(f"Testing RMSE:  {metrics['test_rmse']:.4f}")
    print(f"Testing MAE:   {metrics['test_mae']:.4f}")
    print(f"Data Augmentation: {augment_data}")
    print(f"Stratified Split: {stratify_data}")
    
    return history,  nn_analyzer, metrics, history

# Calling Zone

In [6]:
# Data initiation 
# =============================================================================================================================================================================================
def DatasetCall(ModelSet):
    #Data fixation for the model
    DoNormalize = True
    Z_ScoreScaler = True                       #If false then MinMax Scaler is used in the modelling
    ModelSet = ModelSet                        #1 denotes fixed base (SF all upto 4), 2 denotes SSI only case (SF upto 1 and inputheads include Layers) 3 denotes total datasets, (SF upto 1 , excluding soil layers ie Fixed similar)
    includeCategoricalData = True               #True sets the categorical datta intact and False triggers the one hot encoding to the categorical data
    X, y = dataPreparation(DoNormalize = DoNormalize,Z_ScoreScaler = Z_ScoreScaler, ModelSet = ModelSet ,includeCategoricalData = includeCategoricalData  )
    yscaled = y * 100
    
    X.shape
    # X.columns
    
    # # Execute display of data statistics
    # # =============================================================================================================================================================================================
    
    # # # Generate comprehensive statistics
    # # generate_complete_data_statistics(X, y)
    
    # # # Generate LaTeX table
    # # generate_latex_table(X, y)
    return X, yscaled 

# Calling of Neurons

In [32]:
#__________________________________________________________________________________________________________
ModelSet = 2
X, y  = DatasetCall(ModelSet)

# ##SET A

# 1. Define your hyperparameters (can be tuned via grid search)
nn_params = {
    'architecture': 'wide_regularized',
    'epochs': 500,
    'learning_rate': 0.001,
    'batch_size': 30,
    'augment_data': True,
    'noise_factor': 0.06,
    'stratify_data': True,
    'n_bins': 5,
    'compare_architectures': False
}

# 2. Run the enhanced neural network analysis
nnmodel, nn_analyzer, metrics, history = run_enhanced_neural_network_analysis(
    X, y,
    architecture=nn_params['architecture'],
    compare_architectures=nn_params['compare_architectures'],
    epochs=nn_params['epochs'],
    learning_rate=nn_params['learning_rate'],
    batch_size=nn_params['batch_size'],
    augment_data=nn_params['augment_data'],
    noise_factor=nn_params['noise_factor'],
    stratify_data=nn_params['stratify_data'],
    n_bins=nn_params['n_bins']
)

# Extract the real Keras model from the analyzer
keras_model = nn_analyzer.model

# Extract the history dictionary from the first History object
history_dict = nnmodel.history   # because nnmodel is the History object at index 0

# Metadata (set as needed)
DoNormalize = False
Z_ScoreScaler = None
includeCategoricalData = False

# Save the Keras model as .h5
model_filename = f"NeuralNetwork_Set_{ModelSet}.h5"
keras_model.save(model_filename)   # now works

# Create the joblib package
model_package = {
    "model_path": model_filename,
    "feature_names": X.columns.tolist(),
    "parameters": nn_params,
    "DoNormalize": DoNormalize,
    "Z_ScoreScaler": Z_ScoreScaler,
    "ModelSet": ModelSet,
    "includeCategoricalData": includeCategoricalData,
    "metrics": metrics,
    "history": history_dict          # store the dictionary, not the History object
}

import joblib
joblib.dump(model_package, f"NeuralNetwork_Set_{ModelSet}.joblib")
print("✅ Correctly saved model package")

# ============================================================
# 📊 ENHANCED NEURAL NETWORK RESULTS Set A
# ============================================================
# Training R²:   0.9549
# Testing R²:    0.9346
# Testing RMSE:  0.1430
# Testing MAE:   0.0821
# Data Augmentation: True
# Stratified Split: True

# ============================================================
# 📊 ENHANCED NEURAL NETWORK RESULTS Set B
# ============================================================
# Training R²:   0.9452
# Testing R²:    0.9304
# Testing RMSE:  0.1279
# Testing MAE:   0.0855
# Data Augmentation: True
# Stratified Split: True
# ============================================================
# 📊 ENHANCED NEURAL NETWORK RESULTS Set C
# ============================================================
# Training R²:   0.9591
# Testing R²:    0.9489
# Testing RMSE:  0.1148
# Testing MAE:   0.0753
# Data Augmentation: True
# Stratified Split: True

🚀 Starting Enhanced Neural Network Analysis
📊 Data Augmentation: True
🎯 Stratified Splitting: True
🎯 Using stratified data splitting...
📊 Stratification bins: 5
📈 Target distribution in train: (array([0, 1, 2, 3, 4]), array([404, 403, 404, 403, 404]))
📊 Target distribution in test: (array([0, 1, 2, 3, 4]), array([101, 101, 101, 101, 101]))
🔧 Applying data augmentation with noise factor: 0.06
📈 Data augmented: (2018, 35) -> (4036, 35)
Enhanced Neural Network Analyzer initialized:
Training data: (4036, 35)
Testing data: (505, 35)
Data augmentation: True
Stratified splitting: True
Training Neural Network with wide_regularized architecture...
Epochs: 500, Batch size: 30, Learning rate: 0.001

Epoch 75: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 124: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 182: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 227: ReduceLROnPlateau reducing learning rate to 6.250000296

✅ Correctly saved model package
